# CareerMatch AI - Notebook 03: Agent Definition

**Purpose:** Create and test the CareerMatch AI agent using Unity Catalog tools and MLflow tracing.

## What this notebook does

1. Loads configuration created by the previous notebooks
2. Defines the agent's system prompt, workflow, and guardrails
3. Loads six Unity Catalog functions as callable agent tools
4. Creates the CareerMatch AI agent
5. Tests job search, off-topic handling, vague requests, and the full user-profile workflow
6. Records agent behavior and tool calls with MLflow tracing

## Prerequisites

- Notebook 01 completed and the Vector Search index is available
- Notebook 02 completed and all six Unity Catalog functions are available
- Sample user profiles exist in the database

## Install Dependencies

In [0]:
%pip install databricks-openai mlflow>=2.14.0 openai --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Configuration

In [0]:
# =============================================================================
# CONFIGURATION
# =============================================================================

CATALOG = "finalproject"
SCHEMA = "careermatch_ai"

# LLM endpoint for the agent (orchestrator)
# Options: "databricks-meta-llama-3-3-70b-instruct" or "databricks-gpt-oss-120b"
AGENT_LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

# MLflow experiment name for tracking
MLFLOW_EXPERIMENT_NAME = "/Shared/CareerMatch_AI_Agent"

## Load Configuration

In [0]:
CATALOG = "finalproject"
SCHEMA = "careermatch_ai"
CONFIG_TABLE = f"{CATALOG}.{SCHEMA}.careermatch_config"

config_df = spark.table(CONFIG_TABLE)
config = {row["key"]: row["value"] for row in config_df.collect()}

# Extract key values
JOBS_TABLE_PATH = config["JOBS_TABLE_PATH"]
USER_PROFILES_TABLE = config["USER_PROFILES_TABLE"]
LLM_ENDPOINT = config.get("LLM_ENDPOINT", "databricks-meta-llama-3-3-70b-instruct")

# UC Function names
UC_TOOLS = [
    config["UC_FUNCTION_SEARCH_JOBS"],
    config["UC_FUNCTION_FILTER_JOBS"],
    config["UC_FUNCTION_EXTRACT_SKILLS"],
    config["UC_FUNCTION_GET_USER"],
    config["UC_FUNCTION_MATCH_SCORE"],
    config["UC_FUNCTION_SKILL_RECS"],
]

print("✓ Configuration loaded:")
print(f"  LLM Endpoint: {LLM_ENDPOINT}")
print(f"  UC Tools: {len(UC_TOOLS)} functions")
for tool in UC_TOOLS:
    print(f"    - {tool}")

✓ Configuration loaded:
  LLM Endpoint: databricks-meta-llama-3-3-70b-instruct
  UC Tools: 6 functions
    - finalproject.careermatch_ai.search_jobs
    - finalproject.careermatch_ai.filter_jobs
    - finalproject.careermatch_ai.extract_job_skills
    - finalproject.careermatch_ai.get_user_profile
    - finalproject.careermatch_ai.compute_match_score
    - finalproject.careermatch_ai.get_skill_gap_recommendations


## Imports and MLflow Setup

In [0]:
import json
from typing import Any, Callable, Generator
from uuid import uuid4
import warnings

import mlflow
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
    to_chat_completions_input,
)
from databricks.sdk import WorkspaceClient
from databricks_openai import UCFunctionToolkit
from openai import OpenAI
from pydantic import BaseModel
from unitycatalog.ai.core.base import get_uc_function_client

# Set up MLflow experiment
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

# Enable autologging for tracing
mlflow.openai.autolog()

print(f"✓ MLflow experiment: {MLFLOW_EXPERIMENT_NAME}")

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


✓ MLflow experiment: /Shared/CareerMatch_AI_Agent


## Define System Prompt

The system prompt defines the agent's behavior, available tools, and guardrails.

In [0]:
SYSTEM_PROMPT = """You are CareerMatch AI, an intelligent career matching assistant that helps job seekers find engineering roles that match their skills and experience.

## YOUR CAPABILITIES
You help users by:
1. Finding relevant job postings based on their skills and preferences
2. Calculating match scores between their profile and jobs
3. Identifying skill gaps for desired positions
4. Recommending learning paths to close skill gaps

## AVAILABLE TOOLS
Use these tools to assist users:

- **search_jobs**: Find jobs matching a search query. Use this first to identify candidate jobs.
- **filter_jobs**: Apply filters to job IDs returned by search_jobs. Pass the job IDs as a JSON array, then optionally filter by location, remote status, normalized salary, and work type.
- **extract_job_skills**: Extract required skills from a job description.
- **get_user_profile**: Retrieve a user's profile by their user ID.
- **compute_match_score**: Calculate how well a user matches a specific job (0-100 score).
- **get_skill_gap_recommendations**: Get personalized learning suggestions for missing skills.

## WORKFLOW
For job matching requests, follow this pattern:
1. If user provides a user_id, call get_user_profile to fetch their information
2. Call search_jobs with relevant query based on user's desired role or skills
3. If the user specifies filters, pass the job IDs returned by search_jobs into filter_jobs along with any location, remote, salary, or work-type preferences.
4. For top candidate jobs, call extract_job_skills to get required skills
5. Call compute_match_score to rank jobs against user profile
6. Present top matches with scores and explanations
7. If requested, call get_skill_gap_recommendations for missing skills

## GUARDRAILS
- **Only recommend jobs from search results.** Never invent or hallucinate job postings.
- **Reject off-topic requests.** If the user asks about non-job topics, politely redirect.
- **Handle empty results gracefully.** If search returns no jobs, suggest broadening criteria.
- **Require sufficient information.** If user provides no skills or preferences, ask for more detail.
- **Be transparent about limitations.** If you cannot answer, say so.
- **Do not call tools for vague requests.** If the user does not provide a role, skill, or preference, ask a clarifying question before searching.

## RESPONSE FORMAT
When presenting job matches:
1. List jobs with title, company, location, and match score
2. Explain why each job is a good (or not) match
3. Highlight matching skills and missing skills
4. Offer to provide learning recommendations for skill gaps
5. Convert tool JSON into clear natural-language explanations. Do not show raw JSON or Markdown code fences unless the user asks for them.

## EXAMPLE INTERACTIONS

User: "I'm user_001, find me ML jobs in California"
→ Call get_user_profile('user_001'), then search_jobs('machine learning engineer'), then pass the returned job IDs into filter_jobs with location='CA', then compute_match_score for top results.

User: "What's the weather today?"
→ Politely explain you're a career matching assistant and can help with job searches.

User: "Find me jobs" (no details)
→ Ask for more information: skills, desired role, location preferences, etc.
"""

print("✓ System prompt defined")
print(f"  Length: {len(SYSTEM_PROMPT)} characters")

✓ System prompt defined
  Length: 3245 characters


## Register System Prompt with MLflow

This cell attempts to register the system prompt for version tracking. If schema permissions are unavailable, the notebook continues using the local `SYSTEM_PROMPT`.

In [0]:
from mlflow.genai import register_prompt

# Register the system prompt for versioning
try:
    prompt_info = register_prompt(
        name=f"{CATALOG}.{SCHEMA}.careermatch_system_prompt",
        template=SYSTEM_PROMPT,
        commit_message="Initial CareerMatch AI system prompt with guardrails"
    )
    print(f"✓ System prompt registered: {prompt_info.name}")
except Exception as e:
    print(f"⚠ Prompt registration note: {e}")
    print("  Continuing with local prompt...")

⚠ Prompt registration note: PERMISSION_DENIED: Permission denied to update prompt in schema careermatch_ai. Config: host=https://dbc-c31d04be-d8a2.cloud.databricks.com, auth_type=runtime, retry_timeout_seconds=500
  Continuing with local prompt...


## Define Tool Infrastructure

Set up the tool classes and UC function integration.

In [0]:
class ToolInfo(BaseModel):
    """Represents a tool the agent can call."""
    name: str
    spec: dict
    exec_fn: Callable

    class Config:
        arbitrary_types_allowed = True


def create_tool_info(tool_spec: dict, exec_fn: Callable = None) -> ToolInfo:
    """Create a ToolInfo from a UC function spec."""
    # Remove 'strict' if present (some endpoints don't support it)
    tool_spec.get("function", {}).pop("strict", None)
    tool_name = tool_spec["function"]["name"]
    udf_name = tool_name.replace("__", ".")

    # Default executor calls UC function
    def default_exec_fn(**kwargs):
        parameters = tool_spec.get("function", {}).get("parameters", {})
        required_params = parameters.get("required", [])

        missing_params = [
            param
            for param in required_params
            if kwargs.get(param) is None or kwargs.get(param) == ""
        ]

        if missing_params:
            return (
                "The tool call is missing required information: "
                f"{', '.join(missing_params)}. "
                "Ask the user for the missing details before calling this tool."
            )

        try:
            result = uc_function_client.execute_function(
                udf_name,
                kwargs
            )

            if result.error is not None:
                return f"Tool error: {result.error}"

            return result.value

        except Exception as e:
            return (
                f"Tool execution could not be completed: {e}. "
                "Ask the user for clearer or more complete information."
            )

    return ToolInfo(
        name=tool_name,
        spec=tool_spec,
        exec_fn=exec_fn or default_exec_fn
    )


def clean_schema(schema: Any) -> Any:
    """Remove problematic JSON schema fields that some models reject."""
    if not isinstance(schema, dict):
        return schema

    # Remove format field
    schema.pop("format", None)

    # Recurse into nested schemas
    if "properties" in schema:
        for prop in schema["properties"].values():
            clean_schema(prop)
    if "items" in schema:
        clean_schema(schema["items"])
    for key in ("anyOf", "oneOf", "allOf"):
        if key in schema:
            for item in schema[key]:
                clean_schema(item)

    return schema


def sanitize_tool_spec(spec: dict) -> dict:
    """Sanitize tool spec for compatibility."""
    import copy
    spec = copy.deepcopy(spec)
    params = spec.get("function", {}).get("parameters")
    if params:
        clean_schema(params)
    return spec


print("✓ Tool helpers defined")

✓ Tool helpers defined


/home/spark-13f6f8c1-f377-4689-a757-78/.ipykernel/7912/command-8665764564034223-859358863:1: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class ToolInfo(BaseModel):


## Initialize UC Function Tools

In [0]:
# Initialize UC clients
uc_toolkit = UCFunctionToolkit(function_names=UC_TOOLS)
uc_function_client = get_uc_function_client()

# Create tool infos
TOOL_INFOS = []
for tool_spec in uc_toolkit.tools:
    TOOL_INFOS.append(create_tool_info(tool_spec))

print(f"✓ Loaded {len(TOOL_INFOS)} tools:")
for tool in TOOL_INFOS:
    print(f"  - {tool.name}")

/databricks/python/lib/python3.11/site-packages/databricks/connect/session.py:437: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


✓ Loaded 6 tools:
  - finalproject__careermatch_ai__search_jobs
  - finalproject__careermatch_ai__filter_jobs
  - finalproject__careermatch_ai__extract_job_skills
  - finalproject__careermatch_ai__get_user_profile
  - finalproject__careermatch_ai__compute_match_score
  - finalproject__careermatch_ai__get_skill_gap_recommendations


## Session Cache for Skill Extraction

Caches extracted skills during a session to avoid redundant LLM calls.

In [0]:
class SkillCache:
    """Simple in-memory cache for extracted skills."""

    def __init__(self):
        self._cache = {}

    def get(self, job_id: str) -> dict | None:
        return self._cache.get(str(job_id))

    def set(self, job_id: str, skills: dict):
        self._cache[str(job_id)] = skills

    def clear(self):
        self._cache = {}

    def size(self) -> int:
        return len(self._cache)


# Global session cache
skill_cache = SkillCache()
print("✓ Session cache initialized")

✓ Session cache initialized


## Define the Agent Class

The main agent class that orchestrates tool calls and LLM interactions.

In [0]:
class CareerMatchAgent(ResponsesAgent):
    """
    CareerMatch AI Agent - Helps job seekers find matching engineering roles.

    Uses a ReAct-style pattern:
    1. Reason about what information is needed
    2. Act by calling appropriate tools
    3. Observe results and incorporate into response
    """

    def __init__(self, llm_endpoint: str, tools: list[ToolInfo], system_prompt: str):
        self.llm_endpoint = llm_endpoint
        self.system_prompt = system_prompt
        self.workspace_client = WorkspaceClient()
        self.model_client: OpenAI = (
            self.workspace_client.serving_endpoints.get_open_ai_client()
        )
        self._tools_dict = {tool.name: tool for tool in tools}
        self._skill_cache = SkillCache()

    def get_tool_specs(self) -> list[dict]:
        """Get sanitized tool specifications for the LLM."""
        return [sanitize_tool_spec(tool.spec) for tool in self._tools_dict.values()]

    @mlflow.trace(span_type="TOOL")
    def execute_tool(self, tool_name: str, args: dict) -> Any:
        """Execute a tool call and return the result."""
        # Clean up arguments
        clean_args = {k: v for k, v in (args or {}).items() if k and isinstance(k, str)}

        # Normalize tool name
        name = tool_name.strip().strip('"').strip("'")
        if "<" in name:
            name = name.split("<")[0].strip()

        # Find and execute tool
        if name in self._tools_dict:
            return self._tools_dict[name].exec_fn(**clean_args)

        # Fallback: find by prefix
        candidates = [k for k in self._tools_dict if name.startswith(k)]
        if candidates:
            return self._tools_dict[max(candidates, key=len)].exec_fn(**clean_args)

        raise KeyError(f"Unknown tool: {tool_name}. Available: {list(self._tools_dict.keys())}")

    def call_llm(self, messages: list[dict]) -> Generator[dict, None, None]:
        """Call the LLM with streaming."""
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="PydanticSerializationUnexpectedValue")
            for chunk in self.model_client.chat.completions.create(
                model=self.llm_endpoint,
                messages=to_chat_completions_input(messages),
                tools=self.get_tool_specs(),
                stream=True,
            ):
                chunk_dict = chunk.to_dict()
                if chunk_dict.get("choices"):
                    yield chunk_dict

    def handle_tool_call(
        self,
        tool_call: dict,
        messages: list[dict],
    ) -> ResponsesAgentStreamEvent:
        """Execute a tool call and update message history."""
        try:
            args = json.loads(tool_call.get("arguments", "{}"))
        except json.JSONDecodeError:
            args = {}

        result = str(self.execute_tool(tool_name=tool_call["name"], args=args))

        tool_output = self.create_function_call_output_item(tool_call["call_id"], result)
        messages.append(tool_output)
        return ResponsesAgentStreamEvent(type="response.output_item.done", item=tool_output)

    def call_and_run_tools(
        self,
        messages: list[dict],
        max_iter: int = 15,
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        """Main agent loop: call LLM, execute tools, repeat until done."""
        for _ in range(max_iter):
            last_msg = messages[-1]

            # If last message is assistant response, we're done
            if last_msg.get("role") == "assistant":
                return

            # If last message is a tool call, execute it
            if last_msg.get("type") == "function_call":
                yield self.handle_tool_call(last_msg, messages)
            else:
                # Call LLM and stream response
                yield from output_to_responses_items_stream(
                    chunks=self.call_llm(messages),
                    aggregator=messages
                )

        # Max iterations reached
        yield ResponsesAgentStreamEvent(
            type="response.output_item.done",
            item=self.create_text_output_item(
                "I've reached my maximum number of steps. Please try a more specific request.",
                str(uuid4())
            ),
        )

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        """Handle a complete request (non-streaming)."""
        # Track session in MLflow
        session_id = None
        if request.custom_inputs and "session_id" in request.custom_inputs:
            session_id = request.custom_inputs.get("session_id")
        elif request.context and request.context.conversation_id:
            session_id = request.context.conversation_id

        if session_id:
            mlflow.update_current_trace(metadata={"mlflow.trace.session": session_id})

        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)

    def predict_stream(
        self, request: ResponsesAgentRequest
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        """Handle a streaming request."""
        # Track session
        session_id = None
        if request.custom_inputs and "session_id" in request.custom_inputs:
            session_id = request.custom_inputs.get("session_id")
        elif request.context and request.context.conversation_id:
            session_id = request.context.conversation_id

        if session_id:
            mlflow.update_current_trace(metadata={"mlflow.trace.session": session_id})

        # Build message history with system prompt
        messages = to_chat_completions_input([i.model_dump() for i in request.input])
        messages.insert(0, {"role": "system", "content": self.system_prompt})

        yield from self.call_and_run_tools(messages=messages)


print("✓ CareerMatchAgent class defined")

✓ CareerMatchAgent class defined


## Create Agent Instance

In [0]:
# Create the agent
AGENT = CareerMatchAgent(
    llm_endpoint=AGENT_LLM_ENDPOINT,
    tools=TOOL_INFOS,
    system_prompt=SYSTEM_PROMPT
)

print(f"✓ Agent created")
print(f"  LLM: {AGENT_LLM_ENDPOINT}")
print(f"  Tools: {len(TOOL_INFOS)}")

✓ Agent created
  LLM: databricks-meta-llama-3-3-70b-instruct
  Tools: 6


/home/spark-13f6f8c1-f377-4689-a757-78/.ipykernel/7912/command-8665764564034229-3217559500:16: DeprecationWarning: get_open_ai_client() is deprecated. Please install the databricks-openai package and use 'from databricks_openai import DatabricksOpenAI' instead. See https://api-docs.databricks.com/python/databricks-ai-bridge/latest/databricks_openai.html for more information.
  self.workspace_client.serving_endpoints.get_open_ai_client()


## Test the Agent

Run sample queries to verify the agent works correctly.

### Test 1: Basic Job Search

Test that the agent can search for jobs without a user profile.

In [0]:
from mlflow.types.responses import ResponsesAgentRequest

# Create a test request using OpenAI Responses-style input
test_request = ResponsesAgentRequest(
    input=[
        {
            "role": "user",
            "content": "Find me machine learning engineer jobs in California"
        }
    ]
)

# Run the agent
with mlflow.start_run(run_name="test_basic_search"):
    response = AGENT.predict(test_request)

# Display the response
print("Agent Response:")
print("-" * 50)

for item in response.output:
    if hasattr(item, "content") and item.content:
        print(item.content)
    elif hasattr(item, "text") and item.text:
        print(item.text)
    else:
        print(item)

Agent Response:
--------------------------------------------------
type='function_call' id='chatcmpl_b0f6ba29-3894-4ddd-8f0b-6be4613c8c79' call_id='call_005b5f6c-7cd0-45d4-b905-7a55ea8d35f7' name='finalproject__careermatch_ai__search_jobs' arguments='{ "query": "machine learning engineer California" }'
type='function_call_output' call_id='call_005b5f6c-7cd0-45d4-b905-7a55ea8d35f7' output='job_id,title,company_name,location,formatted_work_type,remote_status,normalized_salary,salary_category,description,search_score\n3904572433,Senior Machine Learning Engineer,Square One Resources,"California, United States",Full-time,Remote,200000.0,120K+,"Senior Machine Learning EngineerSalary: $190,000 - $210,000Location: Based in California, an opportunity to work in one of the tech hubs of the world.\nOur client is in search of a skilled Machine Learning Engineer to develop and enhance the foundational systems that support their product and research teams. This role involves close collaboration with

Trace(trace_id=tr-b732202e938fa7aa4324d8a744eb65ef)

## Basic Job Search Validation

The agent successfully interpreted the request for machine learning engineering jobs in California and called the `search_jobs` tool with an appropriate query.

The returned results included relevant machine learning roles located in California, including remote opportunities. The recommendations were based on records retrieved from the job index rather than invented postings.

This confirms that the agent can perform a basic semantic job search and return relevant results.


### Test 2: Off-Topic Query (Guardrail Test)

Verify the agent rejects non-job queries.

In [0]:
from mlflow.types.responses import ResponsesAgentRequest

test_request_offtopic = ResponsesAgentRequest(
    input=[
        {
            "role": "user",
            "content": "What's the weather like today?"
        }
    ]
)

with mlflow.start_run(run_name="test_guardrail_offtopic"):
    response = AGENT.predict(test_request_offtopic)

print("Agent Response to Off-Topic Query:")
print("-" * 50)

for item in response.output:
    if hasattr(item, "content") and item.content:
        print(item.content)
    elif hasattr(item, "text") and item.text:
        print(item.text)
    else:
        print(item)

Agent Response to Off-Topic Query:
--------------------------------------------------
[{'text': "I'm a career matching assistant, and I don't have information about the weather. I can help you with job searches, provide information about engineering roles, or assist with career development. If you have any questions or need help with something related to your career, feel free to ask!", 'type': 'output_text', 'annotations': []}]


Trace(trace_id=tr-6493d65f5e469f41ee1fc6ec48aa34b2)

## Off-Topic Guardrail Validation

The agent correctly recognized that the weather question was outside the scope of CareerMatch AI.

It did not call any job-search tools and instead politely redirected the user toward career assistance, job searching, and engineering-role questions.

This confirms that the off-topic guardrail worked as intended.


### Test 3: Insufficient Information (Guardrail Test)

Verify the agent asks for more details when query is too vague.

In [0]:
from mlflow.types.responses import ResponsesAgentRequest

test_request_vague = ResponsesAgentRequest(
    input=[
        {
            "role": "user",
            "content": "Find me a job"
        }
    ]
)

with mlflow.start_run(run_name="test_guardrail_vague"):
    response = AGENT.predict(test_request_vague)

print("Agent Response to Vague Query:")
print("-" * 50)

for item in response.output:
    if hasattr(item, "content") and item.content:
        print(item.content)
    elif hasattr(item, "text") and item.text:
        print(item.text)
    else:
        print(item)

Agent Response to Vague Query:
--------------------------------------------------
type='function_call' id='chatcmpl_cfb7a9b3-9138-409f-a12f-a4339da4b710' call_id='call_d3c8859a-4cda-433e-aee3-7faccef15dc3' name='finalproject__careermatch_ai__search_jobs' arguments='{ "query": null }'
type='function_call_output' call_id='call_d3c8859a-4cda-433e-aee3-7faccef15dc3' output='The tool call is missing required information: query. Ask the user for the missing details before calling this tool.'
[{'text': "To find a job that suits you, I need more information. Can you please provide me with some details about the job you're looking for, such as the role, industry, or skills you're interested in? This will help me give you more accurate results.", 'type': 'output_text', 'annotations': []}]


Trace(trace_id=tr-8fcabfef8713e2b3b7b8e373e8b57816)

## Vague-Request Handling Review

The agent partially handled the vague request correctly. It first attempted to call the job-search tool without enough information, which resulted in an incomplete or invalid search request.

After that unsuccessful tool call, the agent asked the user to provide more details about their skills, experience, location, and job preferences.

The final clarification response was appropriate, but the agent should ask for the missing information before calling any tools. This test is therefore considered a partial pass and shows that vague-request handling still needs improvement.

### Test 4: Full Workflow with User Profile

Test the complete flow: user lookup → search → filter → score → recommendations.

**Note:** Requires sample user in `user_profiles` table.

In [0]:
# Check specifically for the sample user used in Test 4
test_user_id = "test_user_001"

test_user_count = (
    spark.table(USER_PROFILES_TABLE)
    .filter(f"user_id = '{test_user_id}'")
    .count()
)

if test_user_count == 0:
    print(f"Creating sample user '{test_user_id}' for testing...")

    spark.sql(f"""
        INSERT INTO {USER_PROFILES_TABLE} VALUES (
            'test_user_001',
            'Alex Johnson',
            'alex@example.com',
            ARRAY('Python', 'SQL', 'Machine Learning', 'TensorFlow', 'Pandas', 'Scikit-learn'),
            'Masters',
            5,
            'Data Scientist',
            'ML Engineer',
            ARRAY('AWS Certified ML Specialty'),
            'Experienced data scientist with 5 years in ML. Built recommendation systems and NLP pipelines. Looking to transition to ML engineering roles with focus on production ML systems.',
            current_timestamp()
        )
    """)

    print("✓ Sample user 'test_user_001' created")
else:
    print("✓ Sample user 'test_user_001' already exists")

display(
    spark.table(USER_PROFILES_TABLE)
    .filter("user_id = 'test_user_001'")
    .select("user_id", "name", "current_role", "desired_role")
)

Creating sample user 'test_user_001' for testing...
✓ Sample user 'test_user_001' created


user_id,name,current_role,desired_role
test_user_001,Alex Johnson,Data Scientist,ML Engineer


In [0]:
from mlflow.types.responses import ResponsesAgentRequest

test_request_full = ResponsesAgentRequest(
    input=[
        {
            "role": "user",
            "content": (
                "I'm test_user_001. Find me ML engineer jobs in California "
                "and tell me which ones I'm best suited for."
            )
        }
    ]
)

with mlflow.start_run(run_name="test_full_workflow"):
    response = AGENT.predict(test_request_full)

print("Agent Response - Full Workflow:")
print("-" * 50)

for item in response.output:
    if hasattr(item, "content") and item.content:
        print(item.content)
    elif hasattr(item, "text") and item.text:
        print(item.text)
    else:
        print(item)

Agent Response - Full Workflow:
--------------------------------------------------
type='function_call' id='chatcmpl_747f84f5-1619-4efb-adf0-5a564d82e4cd' call_id='call_fb13b8db-2da6-430b-afce-b048c709a697' name='finalproject__careermatch_ai__get_user_profile' arguments='{ "input_user_id": "test_user_001" }'
type='function_call_output' call_id='call_fb13b8db-2da6-430b-afce-b048c709a697' output="user_id,name,email,current_skills,education,experience_years,current_role,desired_role,certifications,resume_text\ntest_user_001,Alex Johnson,alex@example.com,['Python' 'SQL' 'Machine Learning' 'TensorFlow' 'Pandas' 'Scikit-learn'],Masters,5,Data Scientist,ML Engineer,['AWS Certified ML Specialty'],Experienced data scientist with 5 years in ML. Built recommendation systems and NLP pipelines. Looking to transition to ML engineering roles with focus on production ML systems.\n"
type='function_call' id='chatcmpl_484d76ff-c441-4e63-9bc8-fbe80565fd28' call_id='call_505d8def-1742-4014-a028-4cdad2a4f5b

Trace(trace_id=tr-d5b1e064532804d3c861dd3e829bede1)

## Full Workflow Review

The agent successfully retrieved the profile for `test_user_001` and used it to search for machine learning engineering jobs in California.

The profile lookup and semantic job search both worked correctly, and the returned jobs were relevant to the user’s target role and location.

However, the complete workflow did not finish. The agent produced a proposed `filter_jobs` call as text instead of executing the tool, and it did not proceed to match scoring, job ranking, or skill-gap recommendations.

This test confirms that profile retrieval and job search are working, but the multi-step tool-calling workflow still needs improvement so the agent consistently executes each required tool.



## Register Agent with MLflow

Log the agent model for versioning and potential deployment.

In [0]:
# Log the agent metadata to MLflow
with mlflow.start_run(run_name="careermatch_agent_v1") as run:
    # Log parameters
    mlflow.log_param("llm_endpoint", AGENT_LLM_ENDPOINT)
    mlflow.log_param("num_tools", len(TOOL_INFOS))
    mlflow.log_param("tool_names", [t.name for t in TOOL_INFOS])

    # Log the system prompt as artifact
    mlflow.log_text(SYSTEM_PROMPT, "system_prompt.txt")

    # Note: Agent contains non-serializable objects (WorkspaceClient)
    # For deployment, use code-based logging or recreate agent in serving context
    mlflow.log_dict(
        {
            "agent_class": "CareerMatchAgent",
            "llm_endpoint": AGENT_LLM_ENDPOINT,
            "system_prompt_length": len(SYSTEM_PROMPT),
            "tools": [t.name for t in TOOL_INFOS]
        },
        "agent_config.json"
    )

    print(f"✓ Agent metadata logged to MLflow")
    print(f"  Run ID: {run.info.run_id}")
    print(f"  Note: Agent instance not serialized (contains non-picklable objects)")

✓ Agent metadata logged to MLflow
  Run ID: 8966222e0b834cf29a8e1ec02882c29c
  Note: Agent instance not serialized (contains non-picklable objects)


## Summary

### Completed

1. ✓ Defined the CareerMatch AI system prompt and guardrails  
2. ✓ Attempted MLflow prompt registration and continued with the local prompt after a permissions error  
3. ✓ Loaded six Unity Catalog functions as agent tools  
4. ✓ Created the `CareerMatchAgent` tool-calling workflow  
5. ✓ Generated MLflow traces for four test scenarios  
6. ✓ Logged agent configuration and metadata to MLflow  

### Test Results

| Test | Scenario | Result |
|---|---|---|
| 1 | Basic job search | Passed — the agent called `search_jobs` and returned relevant California machine-learning roles |
| 2 | Off-topic request | Passed — the agent redirected the user without calling tools |
| 3 | Vague request | Partially passed — the agent asked for clarification after an unsuccessful tool call |
| 4 | Full profile workflow | Partially passed — profile retrieval and job search worked, but filtering, scoring, and skill-gap tools were not completed |

### Remaining Improvements

- Ask for clarification before attempting a tool call when the request lacks necessary details.
- Improve multi-step tool execution so filtering, match scoring, ranking, and skill-gap recommendations complete reliably.
- Use Notebook 04 to evaluate both LLMs systematically and document the agent’s strengths and limitations.

**Next:** Run `04_evaluation.ipynb` to complete the comparative evaluation.

## Export Agent for Notebook 04

In [0]:
# Save agent config for evaluation notebook
agent_config = spark.createDataFrame([
    ("AGENT_LLM_ENDPOINT", AGENT_LLM_ENDPOINT),
    ("MLFLOW_EXPERIMENT_NAME", MLFLOW_EXPERIMENT_NAME),
    ("SYSTEM_PROMPT_LENGTH", str(len(SYSTEM_PROMPT))),
], ["key", "value"])

agent_config.write.mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.careermatch_config")
print("✓ Agent configuration saved for evaluation notebook")

✓ Agent configuration saved for evaluation notebook
